# DATUM large-scale competitive benchmark

This Kaggle notebook runs the reproducible benchmark harness from branch
`benchmark/large-scale-v1`. It records every trial, validates report evidence, and
does not overwrite the `52ea9da` baseline.

The notebook is intentionally explicit: DATUM performs a full build, while pandas,
DuckDB, Polars, and the Python reference perform the declared transform only.
Comparisons are mechanics comparisons, not product rankings.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/datum")
BRANCH = "benchmark/large-scale-v1"
REPO_URL = "https://github.com/mahithpaulin/datum.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, check=True)
os.chdir(REPO)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository:", REPO)
print("branch:", BRANCH)
print("commit:", commit)
print("status:", subprocess.check_output(["git", "status", "--short"], text=True).strip() or "clean")

In [ ]:
import platform

print("python:", sys.version)
print("platform:", platform.platform())
print("machine:", platform.machine())
print("cpu_count:", os.cpu_count())
print("cwd:", os.getcwd())
try:
    with open("/proc/meminfo") as handle:
        for line in handle:
            if line.startswith("MemAvailable:"):
                print("available memory:", line.strip())
except OSError:
    pass
print("disk:", shutil.disk_usage("/kaggle/working"))

In [ ]:
modules = {"pandas": "pandas", "duckdb": "duckdb", "polars": "polars", "psutil": "psutil"}
for module, package in modules.items():
    try:
        imported = __import__(module)
        print(f"{module}: {getattr(imported, '__version__', 'installed')}")
    except Exception as error:
        print(f"{module}: missing ({error}); installing")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

# Kaggle already provides pandas; these are optional benchmark baselines.
subprocess.run([sys.executable, "-m", "pip", "show", "duckdb", "polars", "psutil"], check=False)

In [ ]:
BENCH = Path("/kaggle/working/datum-bench")
RUNS = BENCH / "runs"
RESULTS = BENCH / "results"
RUNS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("runs:", RUNS)
print("results:", RESULTS)

## Tier A: ~10 MiB per profile

All six profiles run with five measured repetitions and one warm-up. Structured
profiles include pandas, DuckDB, Polars, and the independent Python reference when
installed. Large-text, mixed-format, and adversarial profiles are DATUM-only because
their declared transform is not equivalent to the baseline operations.

In [ ]:
PREFLIGHT = BENCH / "preflight"
preflight_profiles = ("csv-transform", "jsonl-training", "many-small-text")
for profile in preflight_profiles:
    command = [
        sys.executable,
        "benchmarks/run.py",
        "--profile", profile,
        "--tier", "A",
        "--target-bytes", "200000",
        "--mode", "equivalent",
        "--tools", "pandas,duckdb,polars",
        "--repetitions", "1",
        "--warmups", "0",
        "--no-verify",
        "--keep-outputs", "first",
        "--runs-dir", str(PREFLIGHT),
        "--run-id", f"preflight-{profile}",
        "--allow-low-resource",
        "--no-hash-fixture",
    ]
    print("\n===", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=REPO, text=True, timeout=900)
    print("exit:", completed.returncode, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"preflight failed for {profile}")
print("preflight passed", flush=True)

In [ ]:
profiles = [
    "many-small-text",
    "large-text",
    "jsonl-training",
    "csv-transform",
    "mixed-formats",
    "adversarial-quality",
]
results = {}
for profile in profiles:
    command = [
        sys.executable,
        "benchmarks/run.py",
        "--profile", profile,
        "--tier", "A",
        "--mode", "both",
        "--tools", "pandas,duckdb,polars",
        "--repetitions", "5",
        "--warmups", "1",
        "--runs-dir", str(RUNS),
        "--run-id", f"kaggle-A-{profile}",
        "--allow-low-resource",
        "--hash-fixture",
    ]
    print("\n===", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=REPO, text=True, timeout=3600)
    results[f"A/{profile}"] = completed.returncode
    print("exit:", completed.returncode, flush=True)
results

## Tier B: ~100 MiB per profile

Tier B repeats the same profiles at 100 MiB. A failure here is kept in the evidence;
it is not hidden or converted into a skip. If the session runs low on time, the
notebook can be rerun with `DATUM_RUN_TIER_B=0`.

In [ ]:
run_tier_b = os.environ.get("DATUM_RUN_TIER_B", "1") == "1"
if run_tier_b:
    for profile in profiles:
        command = [
            sys.executable,
            "benchmarks/run.py",
            "--profile", profile,
            "--tier", "B",
            "--mode", "both",
            "--tools", "pandas,duckdb,polars",
            "--repetitions", "5",
            "--warmups", "1",
            "--keep-outputs", "first",
            "--runs-dir", str(RUNS),
            "--run-id", f"kaggle-B-{profile}",
            "--allow-low-resource",
            "--hash-fixture",
        ]
        print("\n===", " ".join(command), flush=True)
        completed = subprocess.run(command, cwd=REPO, text=True, timeout=7200)
        results[f"B/{profile}"] = completed.returncode
        print("exit:", completed.returncode, flush=True)
else:
    print("Tier B disabled for this session")
results

## Tier C pilot: ~1 GiB

The first 1 GiB runs are pilots for CSV and JSONL. They use three measured
repetitions, no fixture hash, and rely on DATUM's internal verification plus a
separate verifier on the final output. Set `DATUM_RUN_TIER_C=0` to skip the pilot.

In [ ]:
run_tier_c = os.environ.get("DATUM_RUN_TIER_C", "1") == "1"
if run_tier_c:
    for profile in ("csv-transform", "jsonl-training"):
        command = [
            sys.executable,
            "benchmarks/run.py",
            "--profile", profile,
            "--tier", "C",
            "--mode", "both",
            "--tools", "pandas,duckdb,polars",
            "--repetitions", "3",
            "--warmups", "0",
            "--keep-outputs", "first",
            "--runs-dir", str(RUNS),
            "--run-id", f"kaggle-C-{profile}",
            "--allow-low-resource",
            "--no-hash-fixture",
            "--no-verify",
        ]
        print("\n===", " ".join(command), flush=True)
        completed = subprocess.run(command, cwd=REPO, text=True, timeout=14400)
        results[f"C/{profile}"] = completed.returncode
        print("exit:", completed.returncode, flush=True)
        if completed.returncode == 0:
            run_dir = RUNS / f"kaggle-C-{profile}"
            for child in ("fixture", "outputs", "warmups"):
                shutil.rmtree(run_dir / child, ignore_errors=True)
            print("pruned large Tier C artifacts:", run_dir, flush=True)
else:
    print("Tier C disabled for this session")
results

## Validate and aggregate

Every run report is validated independently. The aggregator keeps raw run times,
medians, minima, maxima, IQR, peak RSS, parity, failures, and skipped tools. Compact
reports are copied to `/kaggle/working/datum-bench/results`; generated datasets and
raw tool outputs remain in `runs/` and are not committed to the repository.

In [ ]:
report_paths = sorted(RUNS.glob("*/report.json"))
print("reports:", len(report_paths))
validation = {}
for path in report_paths:
    completed = subprocess.run(
        [sys.executable, "benchmarks/verify_results.py", "--report", str(path), "--json"],
        cwd=REPO,
        text=True,
        capture_output=True,
        timeout=300,
    )
    validation[path.parent.name] = {
        "returncode": completed.returncode,
        "stdout": completed.stdout[-4000:],
        "stderr": completed.stderr[-2000:],
    }
    print(path.parent.name, "valid" if completed.returncode == 0 else "INVALID")

comparison_path = RESULTS / "comparison-large.json"
if report_paths:
    completed = subprocess.run(
        [sys.executable, "benchmarks/compare.py", "--reports", *map(str, report_paths), "--json-out", str(comparison_path)],
        cwd=REPO,
        text=True,
        timeout=300,
    )
    print("aggregate exit:", completed.returncode)
else:
    print("no reports to aggregate")
(RESULTS / "validation.json").write_text(json.dumps(validation, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(RESULTS / "run-results.json").write_text(json.dumps(results, indent=2, sort_keys=True) + "\n", encoding="utf-8")

In [ ]:
copied = []
for path in report_paths:
    run_id = path.parent.name
    target = RESULTS / run_id
    target.mkdir(parents=True, exist_ok=True)
    for name in ("report.json", "report.md", "trials.jsonl", "warmups.jsonl", "fixture.json"):
        source = path.parent / name
        if source.is_file():
            shutil.copy2(source, target / name)
    copied.append(run_id)
    print(run_id, "->", target)

summary = {"copied_runs": copied, "comparison": str(comparison_path)}
(RESULTS / "index.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
if comparison_path.is_file():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    for profile in comparison.get("profiles", []):
        print(f"\n{profile['profile']} tier {profile['tier']} input={profile['input_bytes']} bytes")
        for mode, tools in profile.get("modes", {}).items():
            for tool, entry in tools.items():
                parity = entry.get("parity") or {}
                peak = entry.get("peak_rss_bytes_max")
                print(
                    f"  {mode:10s} {tool:8s} median={entry.get('wall_seconds_median')}s "
                    f"iqr={entry.get('wall_seconds_iqr')}s peak={round(peak / (1024*1024), 1) if peak else None}MiB "
                    f"records={entry.get('records')} parity={parity.get('content_match', '-')}"
                )
    print("\nPreserved failures:", len(comparison.get("failures", [])))
    print("Skipped tools:", len(comparison.get("unsupported", [])))

## Interpretation guardrails

- A single Kaggle machine is not a portable performance guarantee.
- DATUM's wall time includes schema validation, deduplication, quality accounting,
  manifest generation, and post-build verification.
- pandas, DuckDB, Polars, and the Python reference rows are transform-only.
- Report medians with min/max/IQR and raw runs. Do not delete outliers silently.
- Tier D (~5 GiB) and Tier E (~10 GiB) are documented but not attempted here because
  of session, disk, and execution-time limits. Attempting them requires a separate
  resource plan and should start with a pilot.
- The resulting assessment is evidence about mechanics on synthetic data, not proof
  of real-corpus reliability, model quality, or willingness to pay.